# Can we predict 12-month loan default?

Target: `defaulted` (1 = defaulted within 12 months after funding). Plan: audit the data, set a baseline, compare a linear and a tree model with cross-validation, then evaluate once on a held-out test set and check calibration and usefulness for ranking.

In [ ]:
import os; os.environ.setdefault('MPLCONFIGDIR','/tmp/mpl')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, RocCurveDisplay
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
SEED=0
df=pd.read_csv('data.csv')
print(df.shape); df.head()

## 1. Data audit

In [ ]:
print(df.isna().sum().sum(), 'missing;', df.application_id.duplicated().sum(), 'duplicate ids;', df.duplicated().sum(), 'duplicate rows')
display(df.describe().T)
print('Default rate:', df.defaulted.mean().round(4))

In [ ]:
print('employment_years > age:', (df.employment_years > df.age).sum(), '| > age-16:', (df.employment_years > df.age-16).sum(), '(started before 16, mildly implausible)')
print('debt_ratio>1:', (df.debt_ratio>1).sum(), '| non-positive income:', (df.income_k<=0).sum())
df.groupby('region').defaulted.agg(['mean','size'])

`application_id` is an identifier, so it is excluded from the features. There's no application date, so we can't do a time-based split. We use random stratified splits, which may be optimistic if the population drifts over time.

In [ ]:
num=['income_k','debt_ratio','credit_lines','age','employment_years','loan_amount_k']
fig,ax=plt.subplots(2,3,figsize=(13,6))
for a,col in zip(ax.ravel(),num):
    q=pd.qcut(df[col],10,duplicates='drop')
    r=df.groupby(q,observed=True).defaulted.mean()
    a.plot(range(len(r)),r.values,marker='o'); a.axhline(df.defaulted.mean(),ls='--',c='grey')
    a.set_title(f'Default rate by {col} decile'); a.set_xlabel('decile')
plt.tight_layout(); plt.show()

## 2. Split and models
20% held-out test set, untouched until the end. Model selection uses 5-fold stratified CV on the training set. Metrics: ROC AUC (ranking) and average precision (PR AUC), since only ~10% default. Accuracy would be misleading: always predicting 'no default' scores ~90%.

In [ ]:
feat=num+['region']
X,y=df[feat],df.defaulted
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.2,stratify=y,random_state=SEED)
pre=ColumnTransformer([('n',StandardScaler(),num),('c',OneHotEncoder(),['region'])])
models={
 'baseline (prior)':DummyClassifier(strategy='prior'),
 'logistic':make_pipeline(pre,LogisticRegression(max_iter=1000)),
 'gradient boosting':make_pipeline(ColumnTransformer([('c',OneHotEncoder(),['region'])],remainder='passthrough'),HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=SEED)),
}
cv=StratifiedKFold(5,shuffle=True,random_state=SEED)
rows=[]
for n,m in models.items():
    for s in ['roc_auc','average_precision']:
        v=cross_val_score(m,Xtr,ytr,cv=cv,scoring=s)
        rows.append((n,s,v.mean(),v.std()))
pd.DataFrame(rows,columns=['model','metric','mean','std']).pivot(index='model',columns='metric',values=['mean','std']).round(3)

## 3. Held-out test performance

In [ ]:
res={}
for n in ['logistic','gradient boosting']:
    m=models[n].fit(Xtr,ytr); p=m.predict_proba(Xte)[:,1]; res[n]=p
    print(f'{n:18s} AUC={roc_auc_score(yte,p):.3f}  AP={average_precision_score(yte,p):.3f}  Brier={brier_score_loss(yte,p):.4f}')
print(f'base rate AP={yte.mean():.3f}, Brier(prior)={brier_score_loss(yte,np.full(len(yte),ytr.mean())):.4f}')
rng=np.random.default_rng(SEED); p=res['logistic']; yv=yte.values
b=[roc_auc_score(yv[i],p[i]) for i in (rng.integers(0,len(yv),len(yv)) for _ in range(500))]
print('Logistic AUC 95% CI:',np.percentile(b,[2.5,97.5]).round(3))

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,4.5))
for n,p in res.items(): RocCurveDisplay.from_predictions(yte,p,name=n,ax=ax[0])
ax[0].plot([0,1],[0,1],'k--',lw=1); ax[0].set_title('ROC (test)')
for n,p in res.items():
    f,m=calibration_curve(yte,p,n_bins=8,strategy='quantile'); ax[1].plot(m,f,marker='o',label=n)
ax[1].plot([0,.4],[0,.4],'k--',lw=1); ax[1].set_xlabel('predicted'); ax[1].set_ylabel('observed'); ax[1].set_title('Calibration (test)'); ax[1].legend()
plt.tight_layout(); plt.show()

## 4. Is it useful? Risk ranking and drivers

In [ ]:
p=res['logistic']
t=pd.DataFrame({'p':p,'y':yte.values}); t['decile']=pd.qcut(t.p.rank(method='first'),10,labels=range(1,11))
g=t.groupby('decile',observed=True).y.agg(['mean','sum']); g['lift']=g['mean']/yte.mean()
display(g.round(3))
top=t.nlargest(len(t)//5,'p'); print(f'Top 20% riskiest capture {top.y.sum()/t.y.sum():.0%} of defaults')

In [ ]:
lr=models['logistic']
coef=pd.Series(lr[-1].coef_[0],index=lr[0].get_feature_names_out()).sort_values()
print('Logistic coefficients (standardized features):'); display(coef.round(3))
pi=permutation_importance(lr,Xte,yte,scoring='roc_auc',n_repeats=20,random_state=SEED)
pd.Series(pi.importances_mean,index=feat).sort_values().plot.barh(title='Permutation importance (AUC drop, test)'); plt.show()

## 5. Conclusions

**Yes, partially: default is predictable well above chance, but far from precisely.**

- Held-out test: logistic regression AUC ≈ 0.78 (bootstrap 95% CI ≈ 0.745-0.815) against 0.50 for the baseline; average precision ≈ 0.31 against a 0.10 base rate. 5-fold CV on the training set agrees (AUC ≈ 0.775).
- Gradient boosting did not beat logistic regression (CV AUC 0.767 vs 0.775, test 0.756 vs 0.781). The signal looks mostly additive/monotone, so the simpler, more interpretable model is preferred.
- Ranking is useful: the riskiest decile defaults at ~35% (3.4x the base rate); the safest 20% at ~0.6%. The top 20% by score captures ~53% of defaults.
- Drivers (standardized logistic coefficients): higher `debt_ratio` and `loan_amount_k` raise risk; higher `income_k` and `employment_years` lower it. `age`, `credit_lines` and `region` add little.
- It is not a precise per-loan predictor: even in the top decile about 65% of loans do not default.

**Caveats:** random split only (no date column, so drift is untested); ~840 defaults in total, so estimates are noisy, and the test split holds only ~170; 643 rows have `employment_years` > age-16 (3 have it > age), which we left in; associations only, not causal; choose a decision threshold from the actual cost of a missed default versus a rejected good loan. Calibration is shown in section 3 and should be checked before using the probabilities as absolute values.